# COMP34212 Coursework — ResNet-18 Transfer Learning on CIFAR-100 with GradCAM
**Student:** Abdulrahman Bamoukrah | **ID:** 11277638

### Experiments:
1. Baseline: ResNet-18 pretrained, full fine-tune, Adam LR=0.001, BS=64, augmentation ON
2. Low LR (0.0001)
3. High LR (0.01)
4. SGD optimiser
5. Large batch (BS=128)
6. Small batch (BS=32)
7. No data augmentation
8. Frozen backbone (classifier-only)
9. Weight decay L2=0.01
10. ResNet-34 (deeper model)
11. GradCAM visualisation on best model

In [ ]:
# ── Cell 1: Imports ───────────────────────────────────────────────────────────
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
import torchvision.models as models
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import pandas as pd
import copy, os, zipfile
from torch.utils.data import DataLoader, random_split
from PIL import Image

# Check GPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')
if device.type == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')
else:
    print('WARNING: No GPU detected. Go to Runtime > Change runtime type > T4 GPU')

In [ ]:
# ── Cell 2: Dataset & Constants ───────────────────────────────────────────────
N_CLASSES   = 100
N_EPOCHS    = 10       # enough to see convergence
VAL_SPLIT   = 0.1     # 10% of train set used for validation
NUM_WORKERS = 2

# ImageNet normalisation (required for pretrained ResNet)
MEAN = (0.5071, 0.4867, 0.4408)   # CIFAR-100 channel means
STD  = (0.2675, 0.2565, 0.2761)   # CIFAR-100 channel stds

# Augmented transform (used in most experiments)
transform_augment = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.Resize(224),          # ResNet expects 224x224
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD)
])

# Plain transform (no augmentation)
transform_plain = transforms.Compose([
    transforms.Resize(224),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD)
])

# Test transform (always plain)
transform_test = transforms.Compose([
    transforms.Resize(224),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD)
])

def get_loaders(batch_size=64, augment=True):
    """Return train, val, test DataLoaders for CIFAR-100."""
    t_train = transform_augment if augment else transform_plain
    full_train = torchvision.datasets.CIFAR100(
        root='./data', train=True, download=True, transform=t_train)
    test_set = torchvision.datasets.CIFAR100(
        root='./data', train=False, download=True, transform=transform_test)

    n_val   = int(len(full_train) * VAL_SPLIT)
    n_train = len(full_train) - n_val
    train_set, val_set = random_split(
        full_train, [n_train, n_val],
        generator=torch.Generator().manual_seed(42))

    train_loader = DataLoader(train_set, batch_size=batch_size,
                              shuffle=True,  num_workers=NUM_WORKERS, pin_memory=True)
    val_loader   = DataLoader(val_set,   batch_size=batch_size,
                              shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)
    test_loader  = DataLoader(test_set,  batch_size=batch_size,
                              shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)
    return train_loader, val_loader, test_loader

print('Data utilities defined.')

In [ ]:
# ── Cell 3: Model builders ────────────────────────────────────────────────────
def build_resnet18(freeze_backbone=False):
    """ResNet-18 with ImageNet pretrained weights, final layer replaced for CIFAR-100."""
    model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
    if freeze_backbone:
        for param in model.parameters():
            param.requires_grad = False
    # Replace final FC layer
    model.fc = nn.Linear(model.fc.in_features, N_CLASSES)
    return model.to(device)

def build_resnet34():
    """ResNet-34 — deeper model for complexity experiment."""
    model = models.resnet34(weights=models.ResNet34_Weights.IMAGENET1K_V1)
    model.fc = nn.Linear(model.fc.in_features, N_CLASSES)
    return model.to(device)

print('Model builders defined.')

In [ ]:
# ── Cell 4: Training loop ─────────────────────────────────────────────────────
def train_model(model, train_loader, val_loader, optimiser, n_epochs, label):
    """Train model, return history dict with train/val loss and accuracy."""
    criterion = nn.CrossEntropyLoss()
    history = {'train_acc': [], 'val_acc': [], 'train_loss': [], 'val_loss': []}
    best_val_acc = 0.0
    best_weights = copy.deepcopy(model.state_dict())

    print(f'\n{"="*60}')
    print(f'Training: {label}')
    print(f'{"="*60}')

    for epoch in range(n_epochs):
        # ── Train phase ──
        model.train()
        running_loss, correct, total = 0.0, 0, 0
        for inputs, labels in train_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            optimiser.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimiser.step()
            running_loss += loss.item() * inputs.size(0)
            _, predicted = outputs.max(1)
            correct += predicted.eq(labels).sum().item()
            total   += labels.size(0)
        train_loss = running_loss / total
        train_acc  = correct / total

        # ── Val phase ──
        model.eval()
        v_loss, v_correct, v_total = 0.0, 0, 0
        with torch.no_grad():
            for inputs, labels in val_loader:
                inputs, labels = inputs.to(device), labels.to(device)
                outputs = model(inputs)
                loss = criterion(outputs, labels)
                v_loss    += loss.item() * inputs.size(0)
                _, predicted = outputs.max(1)
                v_correct += predicted.eq(labels).sum().item()
                v_total   += labels.size(0)
        val_loss = v_loss / v_total
        val_acc  = v_correct / v_total

        history['train_acc'].append(train_acc)
        history['val_acc'].append(val_acc)
        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_loss)

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_weights = copy.deepcopy(model.state_dict())

        print(f'Epoch [{epoch+1:02d}/{n_epochs}] '
              f'Train Acc: {train_acc:.4f} Loss: {train_loss:.4f} | '
              f'Val Acc: {val_acc:.4f} Loss: {val_loss:.4f}')

    model.load_state_dict(best_weights)  # restore best
    return history

def evaluate(model, test_loader):
    """Return test accuracy and loss."""
    criterion = nn.CrossEntropyLoss()
    model.eval()
    correct, total, running_loss = 0, 0, 0.0
    with torch.no_grad():
        for inputs, labels in test_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            running_loss += loss.item() * inputs.size(0)
            _, predicted = outputs.max(1)
            correct += predicted.eq(labels).sum().item()
            total   += labels.size(0)
    return correct/total, running_loss/total

print('Training loop defined.')

In [ ]:
# ── Cell 5: Plotting helpers ──────────────────────────────────────────────────
def plot_history(histories, labels, filename):
    fig, axes = plt.subplots(2, 2, figsize=(14, 9))
    titles = ['Training Accuracy', 'Validation Accuracy',
              'Training Loss',     'Validation Loss']
    keys   = ['train_acc', 'val_acc', 'train_loss', 'val_loss']
    for ax, title, key in zip(axes.flat, titles, keys):
        for h, lbl in zip(histories, labels):
            ax.plot(h[key], label=lbl)
        ax.set_title(title, fontsize=11, fontweight='bold')
        ax.set_xlabel('Epoch')
        ax.set_ylabel('Accuracy' if 'Acc' in title else 'Loss')
        ax.legend(fontsize=7)
        ax.grid(True)
    plt.tight_layout()
    plt.savefig(filename, dpi=150)
    plt.show()
    print(f'Saved: {filename}')

def bar_chart(names, accs, losses, filename):
    fig, axes = plt.subplots(1, 2, figsize=(15, 5))
    x = range(len(names))
    bars = axes[0].bar(x, [a*100 for a in accs], color='steelblue')
    axes[0].set_xticks(x)
    axes[0].set_xticklabels(names, rotation=35, ha='right', fontsize=8)
    axes[0].set_ylabel('Test Accuracy (%)')
    axes[0].set_title('Test Accuracy Across All Experiments', fontweight='bold')
    axes[0].set_ylim(0, 100)
    for bar, acc in zip(bars, accs):
        axes[0].text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.5,
                     f'{acc*100:.1f}%', ha='center', fontsize=7)
    bars2 = axes[1].bar(x, losses, color='coral')
    axes[1].set_xticks(x)
    axes[1].set_xticklabels(names, rotation=35, ha='right', fontsize=8)
    axes[1].set_ylabel('Test Loss')
    axes[1].set_title('Test Loss Across All Experiments', fontweight='bold')
    for bar, l in zip(bars2, losses):
        axes[1].text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.01,
                     f'{l:.3f}', ha='center', fontsize=7)
    plt.tight_layout()
    plt.savefig(filename, dpi=150)
    plt.show()
    print(f'Saved: {filename}')

print('Plotting helpers defined.')

In [ ]:
# ── Cell 6: GradCAM implementation ───────────────────────────────────────────
class GradCAM:
    """Gradient-weighted Class Activation Mapping for ResNet."""
    def __init__(self, model, target_layer):
        self.model        = model
        self.target_layer = target_layer
        self.gradients    = None
        self.activations  = None
        target_layer.register_forward_hook(self._save_activation)
        target_layer.register_full_backward_hook(self._save_gradient)

    def _save_activation(self, module, input, output):
        self.activations = output.detach()

    def _save_gradient(self, module, grad_input, grad_output):
        self.gradients = grad_output[0].detach()

    def generate(self, input_tensor, class_idx=None):
        self.model.eval()
        output = self.model(input_tensor)
        if class_idx is None:
            class_idx = output.argmax(dim=1).item()
        self.model.zero_grad()
        score = output[0, class_idx]
        score.backward()
        # Pool gradients across spatial dimensions
        weights = self.gradients.mean(dim=[2, 3], keepdim=True)
        cam = (weights * self.activations).sum(dim=1, keepdim=True)
        cam = torch.relu(cam).squeeze().cpu().numpy()
        cam = (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)
        return cam, class_idx

def show_gradcam(model, test_loader, dataset, n_images=8, filename='gradcam.png'):
    """Visualise GradCAM heatmaps on sample test images."""
    # Hook onto the last conv layer of ResNet layer4
    gradcam = GradCAM(model, model.layer4[1].conv2)

    # CIFAR-100 class names
    classes = dataset.classes

    fig, axes = plt.subplots(3, n_images, figsize=(n_images*2, 6))
    fig.suptitle('GradCAM Visualisation — ResNet-18 on CIFAR-100\n'
                 'Row 1: Original | Row 2: GradCAM heatmap | Row 3: Overlay',
                 fontsize=10, fontweight='bold')

    unnorm = transforms.Normalize(
        mean=[-m/s for m,s in zip(MEAN,STD)],
        std=[1/s for s in STD])

    count = 0
    for imgs, lbls in test_loader:
        for i in range(imgs.size(0)):
            if count >= n_images: break
            img_tensor = imgs[i:i+1].to(device)
            cam, pred  = gradcam.generate(img_tensor)

            # Original image (unnormalised)
            orig = unnorm(imgs[i]).permute(1,2,0).numpy().clip(0,1)
            orig_small = Image.fromarray((orig*255).astype(np.uint8)).resize((32,32))

            # Heatmap
            cam_img = Image.fromarray((cm.jet(cam)[:,:,:3]*255).astype(np.uint8)).resize((32,32))

            # Overlay
            cam_resized = np.array(Image.fromarray(
                (cm.jet(cam)[:,:,:3]*255).astype(np.uint8)).resize((32,32)))/255
            overlay = 0.5*orig_small/np.array(orig_small).max() + 0.5*cam_resized
            overlay = (overlay/overlay.max()*255).astype(np.uint8)

            true_lbl = classes[lbls[i].item()]
            pred_lbl = classes[pred]
            correct  = '✓' if pred == lbls[i].item() else '✗'

            axes[0,count].imshow(orig_small)
            axes[0,count].set_title(f'True:\n{true_lbl}', fontsize=6)
            axes[0,count].axis('off')

            axes[1,count].imshow(cam_img)
            axes[1,count].set_title(f'Pred:{correct}\n{pred_lbl}', fontsize=6)
            axes[1,count].axis('off')

            axes[2,count].imshow(overlay)
            axes[2,count].set_title('Overlay', fontsize=6)
            axes[2,count].axis('off')

            count += 1
        if count >= n_images: break

    plt.tight_layout()
    plt.savefig(filename, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'GradCAM saved: {filename}')

print('GradCAM defined.')

In [ ]:
# ── Cell 7: RUN ALL EXPERIMENTS ───────────────────────────────────────────────
# Storage
all_histories = []
all_labels    = []
all_test_acc  = []
all_test_loss = []

def run(label, model, optimiser, batch_size=64, augment=True, n_epochs=N_EPOCHS):
    train_loader, val_loader, test_loader = get_loaders(batch_size, augment)
    h = train_model(model, train_loader, val_loader, optimiser, n_epochs, label)
    acc, loss = evaluate(model, test_loader)
    print(f'>>> [{label}] Test Acc: {acc:.4f} | Test Loss: {loss:.4f}')
    all_histories.append(h)
    all_labels.append(label)
    all_test_acc.append(acc)
    all_test_loss.append(loss)
    return model, test_loader

# ── Exp 1: Baseline ──────────────────────────────────────────────────────────
m1 = build_resnet18()
opt1 = optim.Adam(m1.parameters(), lr=0.001)
best_model, best_test_loader = run('Baseline (Adam, LR=0.001, BS=64, Aug)', m1, opt1)
best_dataset = torchvision.datasets.CIFAR100('./data', train=False,
                   transform=transform_test, download=False)

In [ ]:
# ── Exp 2: Low LR ────────────────────────────────────────────────────────────
m2  = build_resnet18()
opt2 = optim.Adam(m2.parameters(), lr=0.0001)
run('Low LR (0.0001)', m2, opt2)

In [ ]:
# ── Exp 3: High LR ───────────────────────────────────────────────────────────
m3  = build_resnet18()
opt3 = optim.Adam(m3.parameters(), lr=0.01)
run('High LR (0.01)', m3, opt3)

In [ ]:
# ── Exp 4: SGD optimiser ─────────────────────────────────────────────────────
m4  = build_resnet18()
opt4 = optim.SGD(m4.parameters(), lr=0.01, momentum=0.9)
run('SGD (LR=0.01, momentum=0.9)', m4, opt4)

In [ ]:
# ── Exp 5: Large batch ───────────────────────────────────────────────────────
m5  = build_resnet18()
opt5 = optim.Adam(m5.parameters(), lr=0.001)
run('Large Batch (BS=128)', m5, opt5, batch_size=128)

In [ ]:
# ── Exp 6: Small batch ───────────────────────────────────────────────────────
m6  = build_resnet18()
opt6 = optim.Adam(m6.parameters(), lr=0.001)
run('Small Batch (BS=32)', m6, opt6, batch_size=32)

In [ ]:
# ── Exp 7: No augmentation ───────────────────────────────────────────────────
m7  = build_resnet18()
opt7 = optim.Adam(m7.parameters(), lr=0.001)
run('No Augmentation', m7, opt7, augment=False)

In [ ]:
# ── Exp 8: Frozen backbone ───────────────────────────────────────────────────
m8  = build_resnet18(freeze_backbone=True)
opt8 = optim.Adam(filter(lambda p: p.requires_grad, m8.parameters()), lr=0.001)
run('Frozen Backbone (classifier only)', m8, opt8)

In [ ]:
# ── Exp 9: Weight decay ──────────────────────────────────────────────────────
m9  = build_resnet18()
opt9 = optim.Adam(m9.parameters(), lr=0.001, weight_decay=0.01)
run('Weight Decay L2=0.01', m9, opt9)

In [ ]:
# ── Exp 10: ResNet-34 (deeper model) ─────────────────────────────────────────
m10  = build_resnet34()
opt10 = optim.Adam(m10.parameters(), lr=0.001)
run('ResNet-34 (deeper)', m10, opt10)

In [ ]:
# ── Cell 8: Results summary table ────────────────────────────────────────────
df = pd.DataFrame({
    'Experiment':  all_labels,
    'Test Accuracy': [f'{a*100:.2f}%' for a in all_test_acc],
    'Test Loss':   [f'{l:.4f}' for l in all_test_loss]
})
print('\n===== RESULTS SUMMARY =====')
print(df.to_string(index=False))
df.to_csv('results_summary.csv', index=False)
print('\nSaved: results_summary.csv')

In [ ]:
# ── Cell 9: Learning Rate comparison plot ────────────────────────────────────
lr_idx = [0, 1, 2]  # Baseline, Low LR, High LR
plot_history(
    [all_histories[i] for i in lr_idx],
    [all_labels[i] for i in lr_idx],
    'exp_learning_rate.png'
)

In [ ]:
# ── Cell 10: Optimiser comparison plot ───────────────────────────────────────
opt_idx = [0, 3]  # Baseline (Adam) vs SGD
plot_history(
    [all_histories[i] for i in opt_idx],
    [all_labels[i] for i in opt_idx],
    'exp_optimiser.png'
)

In [ ]:
# ── Cell 11: Batch size comparison plot ──────────────────────────────────────
bs_idx = [4, 0, 5]  # BS=128, BS=64, BS=32
plot_history(
    [all_histories[i] for i in bs_idx],
    [all_labels[i] for i in bs_idx],
    'exp_batch_size.png'
)

In [ ]:
# ── Cell 12: Augmentation + Regularisation plot ───────────────────────────────
reg_idx = [0, 6, 7, 8]  # Baseline, No Aug, Frozen, Weight Decay
plot_history(
    [all_histories[i] for i in reg_idx],
    [all_labels[i] for i in reg_idx],
    'exp_regularisation.png'
)

In [ ]:
# ── Cell 13: Depth comparison ─────────────────────────────────────────────────
depth_idx = [0, 9]  # ResNet-18 vs ResNet-34
plot_history(
    [all_histories[i] for i in depth_idx],
    [all_labels[i] for i in depth_idx],
    'exp_depth.png'
)

In [ ]:
# ── Cell 14: Overall bar chart ────────────────────────────────────────────────
bar_chart(all_labels, all_test_acc, all_test_loss, 'all_experiments.png')

In [ ]:
# ── Cell 15: GradCAM visualisation on the best model ─────────────────────────
best_idx = all_test_acc.index(max(all_test_acc))
print(f'Best model: {all_labels[best_idx]} ({all_test_acc[best_idx]*100:.2f}%)')

# Re-evaluate best model GradCAM
# Use the baseline model (m1) which we kept as best_model
show_gradcam(best_model, best_test_loader, best_dataset,
             n_images=8, filename='gradcam_results.png')

In [ ]:
# ── Cell 16: Download everything ──────────────────────────────────────────────
from google.colab import files

to_zip = [f for f in os.listdir('.') if f.endswith(('.png', '.csv'))]
with zipfile.ZipFile('coursework_advanced_results.zip', 'w') as zf:
    for f in to_zip:
        zf.write(f)
        print(f'Added: {f}')

files.download('coursework_advanced_results.zip')
print('\nAll results downloaded!')